In [7]:
import pandas as pd
import numpy as np

# ============================================================
# 1. Load final score files
# ============================================================

true_path = "true_final_scores.csv"
pred_path = "predicted_final_scores.csv"

true_df = pd.read_csv(true_path)
pred_df = pd.read_csv(pred_path)

id_col = "SMILES"
score_col = "final_score"
k = 100

# ============================================================
# 2. Make sure score column is numeric
# ============================================================

true_df[score_col] = pd.to_numeric(true_df[score_col], errors="coerce")
pred_df[score_col] = pd.to_numeric(pred_df[score_col], errors="coerce")

# ============================================================
# 3. Sort by final score and take top 100
# ============================================================

true_sorted = true_df.sort_values(score_col, ascending=False, na_position="last")
pred_sorted = pred_df.sort_values(score_col, ascending=False, na_position="last")

true_top_k = true_sorted.head(k)
pred_top_k = pred_sorted.head(k)

In [8]:
# ============================================================
# 4. Convert top 100 molecules to sets
# ============================================================

true_top_k_set = set(true_top_k[id_col])
pred_top_k_set = set(pred_top_k[id_col])

# ============================================================
# 5. Calculate hits
# ============================================================

hits = true_top_k_set.intersection(pred_top_k_set)
hits_at_k = len(hits)

retrieved_at_k = k
total_relevant = k

# ============================================================
# 6. Calculate Precision@100 and Recall@100
# ============================================================

precision_at_k = hits_at_k / retrieved_at_k
recall_at_k = hits_at_k / total_relevant

# ============================================================
# 7. Calculate Enrichment@100
# ============================================================
# Random baseline = fraction of true top-k molecules in the whole dataset
#
# EF@k = Precision@k / baseline_hit_rate
# baseline_hit_rate = k / N

N = len(true_df)

baseline_hit_rate = k / N
enrichment_at_k = precision_at_k / baseline_hit_rate

# ============================================================
# 8. Print results
# ============================================================

print(f"Total molecules N: {N}")
print(f"Retrieved@{k}: {retrieved_at_k}")
print(f"Hits@{k}: {hits_at_k}")
print(f"Precision@{k}: {precision_at_k:.4f}")
print(f"Recall@{k}: {recall_at_k:.4f}")
print(f"Enrichment@{k}: {enrichment_at_k:.4f}")

Total molecules N: 3000
Retrieved@100: 100
Hits@100: 11
Precision@100: 0.1100
Recall@100: 0.1100
Enrichment@100: 3.3000


In [9]:
# ============================================================
# 9. Save metric result
# ============================================================

metrics_df = pd.DataFrame({
    "k": [k],
    "N": [N],
    f"retrieved@{k}": [retrieved_at_k],
    f"hits@{k}": [hits_at_k],
    f"precision@{k}": [precision_at_k],
    f"recall@{k}": [recall_at_k],
    f"enrichment@{k}": [enrichment_at_k],
    "baseline_hit_rate": [baseline_hit_rate],
})

metrics_df.to_csv(f"ranking_metrics_at_{k}.csv", index=False)

metrics_df

,k,N,retrieved@100,hits@100,precision@100,recall@100,enrichment@100,baseline_hit_rate
0,100,3000,100,11,0.11,0.11,3.3,0.033333
